In [ ]:
import torch
import huggingface_hub

fname = huggingface_hub.hf_hub_download(
    repo_id="ArthurConmy/redwood_attn_2l",
    filename="mask_repeat_candidates.pkl"
)

mask = torch.load(fname, map_location="cpu")

print(type(mask))
print(mask.shape)
print(mask.dtype)
print(mask.requires_grad)

print("True ratio:", mask.float().mean().item())
print("False ratio:", 1 - mask.float().mean().item())

import matplotlib.pyplot as plt

plt.plot(mask[0].numpy())
plt.title("Mask for first example")
plt.show()

data_fname = huggingface_hub.hf_hub_download(
    repo_id="ArthurConmy/redwood_attn_2l",
    filename="validation_data.pt"
)

data = torch.load(data_fname, map_location=torch.device('cpu'))

print(data.shape, mask.shape)

i = 0

plt.figure()
plt.plot(data[i].numpy(), label="tokens")
plt.plot((mask[i] * data[i].max()).numpy(), label="mask (scaled)")
plt.legend()
plt.title("Tokens vs mask positions")
plt.show()


In [ ]:
import torch
import transformers
import os

i = 3

true_positions = torch.where(mask[i])[0]

print("Positions with True:")
print(true_positions.tolist())

os.environ["TRANSFORMERS_USE_FAST"] = "True"

orig_auto = transformers.AutoTokenizer.from_pretrained
orig_gpt2 = transformers.GPT2Tokenizer.from_pretrained
orig_gpt2_fast = transformers.GPT2TokenizerFast.from_pretrained

def universal_tokenizer_patch(orig_fn):
    def patched_fn(pretrained_model_name_or_path, *args, **kwargs):
        if (
            isinstance(pretrained_model_name_or_path, str)
            and "redwood" in pretrained_model_name_or_path.lower()
        ):
            pretrained_model_name_or_path = "gpt2"

        kwargs["use_fast"] = True
        return orig_fn(pretrained_model_name_or_path, *args, **kwargs)

    return patched_fn

transformers.AutoTokenizer.from_pretrained = universal_tokenizer_patch(orig_auto)
transformers.GPT2Tokenizer.from_pretrained = universal_tokenizer_patch(orig_gpt2)
transformers.GPT2TokenizerFast.from_pretrained = universal_tokenizer_patch(orig_gpt2_fast)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    "ArthurConmy/redwood_tokenizer"
)

for p in true_positions:
    p = p.item()

    token_id = int(data[i, p].item())
    token = tokenizer.decode([token_id])

    print(f"Position : {p:4d} | Token id : {token_id:6d} | Token text : {repr(token)}")